# 02. Staged Execution, Multi-Task Tracking & Stateless GCS Resumption

This notebook demonstrates a core design principle of `distillfw`:
- Each task is tracked in an isolated GCS directory (`gs://<bucket>/tasks/<task_id>/`).
- Every stage persists its progress (`task_state.json` + shard/checkpoint cursors) directly to GCS.
- Any worker or VM can resume an interrupted pipeline using **only** the task's GCS URI (`DistillationPipeline.from_task_uri(...)`) with zero local state.

# 02. Staged Execution, Multi-Task Tracking & Stateless GCS Resumption

This notebook demonstrates a core design principle of `distillfw`:
- Each task is tracked in an isolated GCS directory (`gs://<bucket>/tasks/<task_id>/`).
- Every stage persists its progress (`task_state.json` + shard/checkpoint cursors) directly to GCS.
- Any worker or VM can resume an interrupted pipeline using **only** the task's GCS URI (`DistillationPipeline.from_task_uri(...)`) with zero local state.

In [ ]:
import os, json, tempfile
from pathlib import Path
from distillfw import DistillationConfig, DistillationPipeline, GCPConfig, StageName

with tempfile.TemporaryDirectory() as tmp:
    os.environ["DISTILLFW_LOCAL_GCS_ROOT"] = tmp
    prompts_file = Path(tmp) / "prompts.jsonl"
    prompts_file.write_text("\n".join(json.dumps({"prompt": f"Prompt {i}"}) for i in range(12)))

    cfg = DistillationConfig(
        task_id="resumable-task-demo",
        gcp=GCPConfig(project_id="demo-proj", bucket_name="demo-bucket"),
    )

    # Step 1: Initialize task and stop after Stage 2 (dataset_formatter)
    p1 = DistillationPipeline.init_task(
        config=cfg,
        prompts_path=prompts_file,
        teacher_callable=lambda p, _: {"completion": f"Teacher answer for {p}"},
    )
    p1.resume(stop_after=StageName.DATASET_FORMATTER)
    print("After partial run on Machine A:", p1.workspace.next_pending_stage())

    # Step 2: Attach from a completely stateless process on Machine B using ONLY task_uri
    p2 = DistillationPipeline.from_task_uri(
        cfg.task_uri,
        custom_train_fn=lambda tp, ckpt, exp: {"train_loss": 0.25, "global_step": 10},
        student_predict_fn=lambda ps: (["Student output"] * len(ps), [12.0] * len(ps)),
        judge_fn=lambda p, r, s: {"score": 4, "reason": "Good"},
        deploy_fn=lambda tid, uri, dcfg: {"endpoint_resource_name": f"endpoints/{tid}"},
    )
    final_state = p2.resume()
    print("After resuming on Machine B from GCS URI alone:", final_state.status)

In [ ]:
import os, json, tempfile
from pathlib import Path
from distillfw import DistillationConfig, DistillationPipeline, GCPConfig, StageName

with tempfile.TemporaryDirectory() as tmp:
    os.environ["DISTILLFW_LOCAL_GCS_ROOT"] = tmp
    prompts_file = Path(tmp) / "prompts.jsonl"
    prompts_file.write_text("\n".join(json.dumps({"prompt": f"Prompt {i}"}) for i in range(12)))

    cfg = DistillationConfig(
        task_id="resumable-task-demo",
        gcp=GCPConfig(project_id="demo-proj", bucket_name="demo-bucket"),
    )

    # Step 1: Initialize task and stop after Stage 2 (dataset_formatter)
    p1 = DistillationPipeline.init_task(
        config=cfg,
        prompts_path=prompts_file,
        teacher_callable=lambda p, _: {"completion": f"Teacher answer for {p}"},
    )
    p1.resume(stop_after=StageName.DATASET_FORMATTER)
    print("After partial run on Machine A:", p1.workspace.next_pending_stage())

    # Step 2: Attach from a completely stateless process on Machine B using ONLY task_uri
    p2 = DistillationPipeline.from_task_uri(
        cfg.task_uri,
        custom_train_fn=lambda tp, ckpt, exp: {"train_loss": 0.25, "global_step": 10},
        student_predict_fn=lambda ps: (["Student output"] * len(ps), [12.0] * len(ps)),
        judge_fn=lambda p, r, s: {"score": 4, "reason": "Good"},
        deploy_fn=lambda tid, uri, dcfg: {"endpoint_resource_name": f"endpoints/{tid}"},
    )
    final_state = p2.resume()
    print("After resuming on Machine B from GCS URI alone:", final_state.status)